<h1><center><strong></strong></center></h1>
<h1><center><strong>CME466</strong></center></h1>
<h1><center><strong>Design of an Advanced Digital System</strong></center></h1>
<p><center><strong>Department of Electrical and Computer Engineering</strong></center></p>
<p><center><strong>2025 Winter Term</strong></center></p>

# Deliverable 5: CIFAR-10 Classification

This notebook serves as a template for your homework assignment. It is based on the notebooks we used in class,
but you will need to make some modifications to adapt it to the CIFAR-10 dataset.

### About the Dataset:
CIFAR-10 is a dataset consisting of 60,000 32x32 color images belonging to 10 different classes:
['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

### What You Need to Do:
- Most of the code structure remains the same as in previous notebooks.
- However, you must modify certain parts, such as model architecture, to handle CIFAR-10 images properly.
- Functions like `train_step`, `test_step`, and `visualize_training` are provided to help you, but they will work correctly
  only if you follow the procedures we discussed in class.

Make sure to carefully read the provided code and modify the necessary sections to complete the classification task.

Complete the required sections of the functions wherever you see the following comment:
<p><code># >>> ENTER YOUR CODE HERE <<<
</code></p>



# 0. Prerequisites

## 0.1 Import Libraries

In [ ]:
# Import PyTorch
import torch
from torch import nn

# Import torchvision
import torchvision
from torchvision import datasets
from torchvision.transforms import ToTensor

# Import matplotlib for visualization
import matplotlib.pyplot as plt
from pathlib import Path

# Check versions
print(f"PyTorch version: {torch.__version__}\ntorchvision version: {torchvision.__version__}")

In [ ]:
# Try to import torchinfo, install if it doesn't work
try:
    from torchinfo import summary
except:
    print("[INFO] coudln't find torchinfo, installing it...")
    !pip install -q torchinfo
    from torchinfo import summary
    print("[INFO] Done!")

In [ ]:
# Try to import torchmetrics and install if it doesn't work
try:
    import torchmetrics
except:
    print("[INFO] coudln't find torchmetrics, installing it...")
    !pip install -q torchmetrics
    import torchmetrics
    print("[INFO] Done!")

In [ ]:
from torchmetrics.classification import MulticlassAccuracy

## 0.2 Setup Device agnostic code

In [ ]:
# Set up device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
%matplotlib inline

# 1. Downloading and Transforming the CIFAR-10 Dataset

![CIFAR10 Dataset](https://pytorch.org/tutorials/_images/cifar10.png)


Before proceeding, refer to the <a href="https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html">**CIFAR10 Tutorial**</a> link to understand the necessary transformations for this dataset.
Additionally, check the <a href="https://pytorch.org/vision/main/generated/torchvision.datasets.CIFAR10.html">**CIFAR10**</a> link to learn about the function required to download the dataset.

### Instructions:
1. Define a transformation pipeline that:
   - Converts images to tensors.
   - Normalizes the images properly.
2. Use the appropriate function to:
   - Download the CIFAR-10 dataset.
   - Apply the transformations.
   - Set up separate datasets for training and testing.

Make sure to implement this step correctly before proceeding with model training.


In [ ]:
import torchvision.transforms as transforms

# >>> ENTER YOUR CODE HERE <<<
# 1. Define the necessary transformations (e.g., ToTensor, Normalize)
transform = transforms.Compose([
    # Add the transformations here
])

# >>> ENTER YOUR CODE HERE <<<
# 2. Download the CIFAR-10 training dataset and apply the transformations
# Make sure to use the correct function to download the dataset and set the 'train' parameter to True
train_data =

# >>> ENTER YOUR CODE HERE <<<
# 3. Download the CIFAR-10 test dataset and apply the transformations
# Make sure to set the 'train' parameter to False for the test data
test_data =

## 1.2 Image Shapes


Let's check out the first sample of the training data.

In [ ]:
# See first training sample
image, label = train_data[0]
image, label

Find out about a single image shape from this dataset. how many colour channels we have in here?

In [ ]:
image.shape

In [ ]:
# How many samples are there?
len(train_data.data), len(train_data.targets), len(test_data.data), len(test_data.targets)

What classes are there?

In [ ]:
# See classes
# >>> ENTER YOUR CODE HERE <<<
class_names =
class_names

## 1.3 Visualizing the CIFAR-10 Images

<p>There is a subtle difference between visualizing FashionMNIST images and CIFAR-10 images.</p>
<p>CIFAR-10 images are colored and have a different shape, so the visualization technique will vary slightly.</p>

Refer to the <a href="https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html">**CIFAR10 Tutorial**</a> to find the correct way to visualize the CIFAR-10 images.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

image, label = train_data[0]
print(f"Image shape: {image.shape}")
print(f"Label: {class_names[label]}")

# >>> ENTER YOUR CODE HERE <<<



plt.title(label)
plt.show()

In [ ]:
# Plot more images
torch.manual_seed(42)
fig = plt.figure(figsize=(9, 9))
rows, cols = 4, 4
for i in range(1, rows * cols + 1):
    random_idx = torch.randint(0, len(train_data), size=[1]).item()
    img, label = train_data[random_idx]
    fig.add_subplot(rows, cols, i)

    # >>> ENTER YOUR CODE HERE <<<



    plt.title(class_names[label])
    plt.axis(False);

# 2. Prepare DataLoader

**No Changes Needed for This Part**

Thanks to the modular and clean code we've written, we don't need to change this part :)

In [ ]:
from torch.utils.data import DataLoader

# Setup the batch size hyperparameter
BATCH_SIZE = 32

# Turn datasets into iterables (batches)
train_dataloader = DataLoader(train_data, # dataset to turn into iterable
    batch_size=BATCH_SIZE, # how many samples per batch?
    shuffle=True # shuffle data every epoch?
)

test_dataloader = DataLoader(test_data,
    batch_size=BATCH_SIZE,
    shuffle=False # don't necessarily have to shuffle the testing data
)

# Let's check out what we've created
print(f"Dataloaders: {train_dataloader, test_dataloader}")
print(f"Length of train dataloader: {len(train_dataloader)} batches of {BATCH_SIZE}")
print(f"Length of test dataloader: {len(test_dataloader)} batches of {BATCH_SIZE}")

In [ ]:
# Check out what's inside the training dataloader
train_features_batch, train_labels_batch = next(iter(train_dataloader))
train_features_batch.shape, train_labels_batch.shape

And we can see that the data remains unchanged by checking a single sample.

In [ ]:
# Show a sample
torch.manual_seed(42)
random_idx = torch.randint(0, len(train_features_batch), size=[1]).item()
img, label = train_features_batch[random_idx], train_labels_batch[random_idx]

# >>> ENTER YOUR CODE HERE <<<



plt.title(class_names[label])
plt.axis("Off");
print(f"Image size: {img.shape}")
print(f"Label: {label}")

# 3. Helper Functions

<p>I have provided several helper functions to make your task easier. These functions are modular and can be reused throughout the notebook.</p>
<p>They handle common tasks such as training steps, testing steps, and visualizing the training process.</p>

<p>You do not need to modify these functions unless specified. Simply use them as instructed in the notebook.</p>


## 3.1 Training Step Function

I have provided the `train_step` function to handle a single step of training the model. This function computes the loss,
performs the backpropagation step, and updates the model weights.

### Function Explanation:
- `model`: The model that you are training.
- `data_loader`: The dataset used for training, wrapped in a DataLoader.
- `loss_fn`: The loss function used to compute the error between the predicted and actual labels.
- `optimizer`: The optimizer responsible for adjusting the model's weights based on gradients.
- `num_classes`: The number of classes in the dataset (e.g., 10 for CIFAR-10).
- `device`: The device (CPU or GPU) where the computations will be performed.

The function executes the following steps during training:
1. **Forward Pass**: The model makes predictions on the input batch.
2. **Loss Calculation**: The loss function computes the error between the predicted labels and true labels.
3. **Zero Gradients**: The gradients are reset to prepare for the next update.
4. **Backpropagation**: The gradients are calculated through backpropagation.
5. **Optimizer Step**: The optimizer updates the model weights based on the gradients.

At the end of the function, it prints the training loss and accuracy for that batch.

### Usage:
You don't need to modify this function. Simply call it in the training loop for each batch to update the model weights.

Example:
```python
train_loss, train_acc = train_step(model, train_loader, loss_fn, optimizer, num_classes, device)


In [ ]:
def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               num_classes,
               device: torch.device = device):
    # Instantiate Multiclass Accuracy instance
    acc = MulticlassAccuracy(num_classes=num_classes, average='micro').to(device)
    train_loss = 0
    model.to(device)
    for batch, (X, y) in enumerate(data_loader):
        # Send data to GPU
        X, y = X.to(device), y.to(device)

        # 1. Forward pass
        y_logits = model(X)

        # 2. Calculate loss
        loss = loss_fn(y_logits, y)
        train_loss += loss
        y_pred_prob = torch.softmax(y_logits, dim=1)
        y_pred_labels = torch.argmax(y_pred_prob, dim=1)
        acc.update(y_pred_labels, y)

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()

    # Calculate loss and accuracy per epoch and print out what's happening
    train_loss /= len(data_loader)
    train_acc = acc.compute().cpu().numpy()

    acc.reset()

    print(f"Train loss: {train_loss:.5f} | Train accuracy: {train_acc:.2f}%")

    return train_loss, train_acc

## 3.2 Test Step Function

The `test_step` function evaluates the performance of the trained model on the testing dataset. It computes the loss
and accuracy of the model during testing and prints the results.

### Function Explanation:
- `data_loader`: The testing dataset wrapped in a DataLoader.
- `model`: The trained model to evaluate.
- `loss_fn`: The loss function used to compute the error between the predicted and actual labels.
- `num_classes`: The number of classes in the dataset (e.g., 10 for CIFAR-10).
- `device`: The device (CPU or GPU) where the computations will be performed.

The function follows these steps during testing:
1. **Forward Pass**: The model makes predictions on the input batch.
2. **Loss Calculation**: The loss function computes the error between the predicted labels and true labels.
3. **Accuracy Calculation**: It computes the accuracy by comparing the predicted labels to the true labels.
4. **Results**: The function prints the average loss and accuracy after testing on the entire dataset.

At the end of the function, it prints the test loss and accuracy for the entire dataset.

### Usage:
You don't need to modify this function. Simply call it after completing each training epoch to evaluate the model's performance on the test dataset.

Example:
```python
test_loss, test_acc = test_step(test_loader, model, loss_fn, num_classes, device)


In [ ]:
def test_step(data_loader: torch.utils.data.DataLoader,
              model: torch.nn.Module,
              loss_fn: torch.nn.Module,
              num_classes,
              device: torch.device = device):

    acc = MulticlassAccuracy(num_classes=num_classes, average='micro').to(device)
    test_loss = 0
    model.to(device)
    model.eval() # put model in eval mode
    # Turn on inference context manager
    with torch.inference_mode():
        for X, y in data_loader:
            # Send data to GPU
            X, y = X.to(device), y.to(device)

            # 1. Forward pass
            test_logits = model(X)

            # 2. Calculate loss and accuracy
            test_loss += loss_fn(test_logits, y)
            test_pred_probs = torch.softmax(test_logits, dim=1)
            test_pred_labels = torch.argmax(test_pred_probs, dim=1)
            acc.update(test_pred_labels, y)

        # Adjust metrics and print out
        test_loss /= len(data_loader)
        test_acc = acc.compute().cpu().numpy()

        acc.reset()
        print(f"Test loss: {test_loss:.5f} | Test accuracy: {test_acc:.2f}%\n")
        return test_loss, test_acc

## 3.3 Model Evaluation Function

I have provided the `eval_model` function to evaluate your model's performance. This function calculates both the loss
and accuracy of the model on the given dataset.

### Function Explanation:
- `model`: The trained PyTorch model that you want to evaluate.
- `data_loader`: The dataset that you want to evaluate the model on.
- `loss_fn`: The loss function used for evaluating the model’s performance.
- `num_classes`: The number of classes in your dataset (CIFAR-10 has 10 classes).
- `device`: The device (CPU or GPU) where the evaluation will be performed. By default, it's set to use the device you specified earlier.

The function returns a dictionary with:
- `model_name`: The name of the model class.
- `model_loss`: The loss value calculated during the evaluation.
- `model_acc`: The accuracy of the model on the evaluation dataset.

### Usage:
You don't need to modify this function. Simply call it after training your model to evaluate its performance.

Example:
```python
eval_results = eval_model(model, test_loader, loss_fn, num_classes, device)
print(eval_results)


In [ ]:
# Move values to device
torch.manual_seed(42)
def eval_model(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               num_classes,
               device: torch.device = device):
    """Evaluates a given model on a given dataset.

    Args:
        model (torch.nn.Module): A PyTorch model capable of making predictions on data_loader.
        data_loader (torch.utils.data.DataLoader): The target dataset to predict on.
        loss_fn (torch.nn.Module): The loss function of model.
        num_classes (int): Number of classes in dataset.
        device (str, optional): Target device to compute on. Defaults to device.

    Returns:
        (dict): Results of model making predictions on data_loader.
    """
    acc = MulticlassAccuracy(num_classes=num_classes, average='macro').to(device)
    loss = 0
    model.eval()
    with torch.inference_mode():
        for X, y in data_loader:
            # Send data to the target device
            X, y = X.to(device), y.to(device)
            y_logits = model(X)
            loss += loss_fn(y_logits, y)
            y_pred_probs = torch.softmax(y_logits, dim=1)
            y_pred_labels = torch.argmax(y_pred_probs, dim=1)
            acc.update(y_pred_labels, y)

        # Scale loss and acc
        loss /= len(data_loader)
        test_acc = acc.compute().cpu().numpy()
        acc.reset()

    return {"model_name": model.__class__.__name__, # only works when model was created with a class
            "model_loss": loss.item(),
            "model_acc": test_acc.item()}

## 3.4 Visualizing Training Results

The `visualize_training` function plots the training and testing loss as well as the training and testing accuracy over multiple epochs.
This allows you to visually track the model's performance during training.

### Function Explanation:
- `results`: A dictionary containing the training and testing loss and accuracy values. It should have the following structure:
    - `"train_loss"`: A list of training loss values for each epoch.
    - `"test_loss"`: A list of test loss values for each epoch.
    - `"train_acc"`: A list of training accuracy values for each epoch.
    - `"test_acc"`: A list of test accuracy values for each epoch.

The function generates two plots:
1. **Loss Plot**: Shows the training and test loss curves. This helps to track how well the model is minimizing the loss during training and generalizing during testing.
2. **Accuracy Plot**: Shows the training and test accuracy curves. This helps to track the model's performance in terms of correct predictions over epochs.

The plots will automatically scale to the appropriate ranges for both loss and accuracy, and a grid is added for better readability.

### Usage:
After training your model, you can call this function to visualize the results:

```python
visualize_training(results)


In [ ]:
import matplotlib.pyplot as plt
def visualize_training(results):

    # Plot loss
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(range(1, len(results["train_loss"]) + 1), results["train_loss"], 'bo-', label='Train Loss')
    plt.plot(range(1, len(results["train_loss"]) + 1), results["test_loss"], 'ro-', label='Test Loss')
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Training & Testing Loss")
    plt.legend()
    plt.grid()
    plt.ylim(0, max(max(results["train_loss"]), max(results["test_loss"])) * 1.1)  # Ensure the loss starts from 0

    # Plot accuracy
    plt.subplot(1, 2, 2)
    plt.plot(range(1, len(results["train_loss"]) + 1), results["train_acc"], 'bo-', label='Train Accuracy')
    plt.plot(range(1, len(results["train_loss"]) + 1), results["test_acc"], 'ro-', label='Test Accuracy')
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy (%)")
    plt.title("Training & Testing Accuracy")
    plt.legend()
    plt.grid()
    plt.ylim(0, 1)  # Accuracy should be between 0% and 100%

    plt.show()

##3.5 Plotting Predictions

These two functions help visualize the model's predictions and the corresponding probability values for CIFAR-10 images.
They provide insights into how well the model is classifying the images and whether it is confident in its predictions.

### Function Explanation:
1. **`plot_image(i, predictions_array, true_label, img)`**:
   - **Purpose**: Displays a single image with the predicted label, the true label, and the prediction probability.
   - **Parameters**:
     - `i`: The index of the image in the batch.
     - `predictions_array`: The model's output probabilities for the image (before applying `argmax`).
     - `true_label`: The true label of the image.
     - `img`: The image tensor, which will be plotted.
   - **What It Does**:
     - This function unnormalizes the image (scaling it back to the range `[0, 1]`), displays it, and overlays the predicted and true labels with their respective confidence levels.
     - If the predicted label matches the true label, it is displayed in blue; otherwise, in red.

2. **`plot_value_array(i, predictions_array, true_label)`**:
   - **Purpose**: Displays a bar chart of the prediction probabilities for each class, highlighting the predicted and true labels.
   - **Parameters**:
     - `i`: The index of the image in the batch.
     - `predictions_array`: The model's output probabilities for the image (before applying `argmax`).
     - `true_label`: The true label of the image.
   - **What It Does**:
     - This function creates a bar chart showing the prediction probabilities for each of the 10 classes (CIFAR-10 has 10 classes). The predicted label is shown in red, while the true label is shown in blue.

### Usage:
You can use these functions to visualize individual predictions after testing the model:

```python
plot_image(i, predictions_array, true_label, img)
plot_value_array(i, predictions_array, true_label)


In [ ]:
# Plot Predictions
import numpy as np

def plot_image(i, predictions_array, true_label, img):
    true_label, img = true_label[i], img[i]
    plt.grid(False)
    plt.xticks([])
    plt.yticks([])

    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))

    predicted_label = torch.argmax(predictions_array).item()

    if predicted_label == true_label:
        color = 'blue'
    else:
        color = 'red'

    plt.xlabel("{} {:2.0f}% ({})".format(class_names[predicted_label],
                                100*torch.max(predictions_array).item(),
                                class_names[true_label]),
                                color=color)

def plot_value_array(i, predictions_array, true_label):
  true_label = true_label[i]
  plt.grid(False)
  plt.xticks(range(10))
  plt.yticks([])
  thisplot = plt.bar(range(10), predictions_array, color="#777777")
  plt.ylim([0, 1])
  predicted_label = np.argmax(predictions_array)

  thisplot[predicted_label].set_color('red')
  thisplot[true_label].set_color('blue')

# 4. Model 1: Building a multi-class clsification model

Before we dive into building the model, let's address an important point. You might have noticed that the CIFAR-10 image
shape is different from the FashionMNIST images we worked with earlier. This difference in shape means we need to adjust
our model accordingly.

The first hidden layer of your model should be compatible with the shape of the CIFAR-10 images. To determine the
correct input shape for the first hidden layer, execute the following cell. This will help you understand how to
properly design your model architecture.


In [ ]:
# Create a flatten layer
flatten_model = nn.Flatten() # all nn modules function as a model (can do a forward pass)

# Get a single sample
x = train_features_batch[0]

# Flatten the sample
output = flatten_model(x) # perform forward pass

# Print out what happened
print(f"Shape before flattening: {x.shape} -> [color_channels, height, width]")
print(f"Shape after flattening: {output.shape} -> [color_channels, height*width]")

## 4.1 Building the Model

Now, it's time to design our model! Based on what we've learned, we need to build a model that can classify CIFAR-10 images.

### Key Considerations:
- **Input Shape**: As you've seen, the input images are 32x32x3 (RGB), so make sure your model’s first layer can handle this input shape.
- **Layers**: You need to create an architecture that can learn the patterns from these images. You can use fully connected (linear) layers, activation functions (like ReLU), and output layers. Remember that CIFAR-10 has 10 classes, so your final output layer should have 10 units.
- **Model Design**: You have the freedom to design the model. You can experiment with the number of layers, units per layer, and activation functions. Start simple and feel free to experiment with different architectures!

### Next Steps:
- **Input Layer**: Consider how to flatten the 32x32x3 input into a 1D tensor that can be passed into the linear layers.
- **Hidden Layers**: Add multiple hidden layers with activation functions. You can experiment with the number of layers and the number of neurons per layer.
- **Output Layer**: Your model will classify into one of 10 categories, so the output layer should have 10 units.

In [ ]:
# Create a model with non-linear and linear layers
class CIFAR10ModelV1(nn.Module):
    def __init__(self, input_shape: int, output_shape: int):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Flatten(), # flatten inputs into single vector
            # >>> ENTER YOUR CODE HERE <<<
        )

    def forward(self, x: torch.Tensor):
        return self.layer_stack(x)

## 4.2 Initialize the Model and Check Device

In this step, you need to create an instance of the model you designed in the previous cell.

### Key Instructions:
- **Model Instance**: You need to create an instance of your model by passing the appropriate input and output shapes.
- **Device Handling**: Make sure that the model is placed on the right device (GPU or CPU). You can do this by moving the model to the device you are using for training. You should already know the device from the earlier cells.
- **Check Model Device**: After creating the model, we will check that the model is properly loaded onto the right device.

### Steps:
1. Create an instance of the model and ensure it’s moved to the correct device (e.g., `.to(device)`).
2. Check that the model is on the correct device by printing the device of one of the model’s parameters.

In [ ]:
torch.manual_seed(42)
# Create an instance of your model
# also make sure the model is on the right device
 # >>> ENTER YOUR CODE HERE <<<
model_1 =


next(model_1.parameters()).device # check model device

How many parameters does your model have?

In [ ]:
from torchinfo import summary
summary(model=model_1,
         # >>> ENTER YOUR CODE HERE <<<
        input_size=,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        row_settings=["var_names"])

## 3.1 Setup loss, optimizer and evaluation metrics

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_1.parameters(),
                            lr=0.001)

## 3.2 Training the model

If you have followed the instructions in the previous steps, there's no need to modify anything in this cell.
This step will automatically train the model that you have created and moved to the correct device.

### What Happens in This Step:
- The model will be trained using the training data you set up earlier.
- The training process will be monitored, and the loss and accuracy will be printed out.
- You don't need to change anything here unless you want to experiment with different settings or configurations.

### Note:
- The code here will automatically use the `train_step` function and the provided training data loader to train your model.

So, simply run the cell to begin training your model. If you've followed the steps correctly, you're all set!


In [ ]:
# Import tqdm for progress bar
from tqdm.auto import tqdm
torch.manual_seed(42)

epochs = 10

# Create empty results dictionary
results = {"train_loss": [],
           "train_acc": [],
           "test_loss": [],
           "test_acc": []
}

for epoch in tqdm(range(epochs)):
    print(f"Epoch: {epoch}\n---------")
    train_loss, train_acc = train_step(data_loader=train_dataloader,
                                       model=model_1,
                                       loss_fn=loss_fn,
                                       optimizer=optimizer,
                                       num_classes=len(class_names)
                                    )
    test_loss, test_acc = test_step(data_loader=test_dataloader,
                                    model=model_1,
                                    loss_fn=loss_fn,
                                    num_classes=len(class_names)
                                )

    # Update results dictionary
    results["train_loss"].append(train_loss.detach().cpu().numpy())
    results["train_acc"].append(train_acc)
    results["test_loss"].append(test_loss.detach().cpu().numpy())
    results["test_acc"].append(test_acc)

## 3.3 Visualizing results

In this step, we will visualize the training and testing results. You don't need to modify anything in this cell. If you have followed the previous instructions, the training and testing data will already be available, and this cell will automatically plot the loss and accuracy curves.

### What Happens in This Step:
- **Loss Curve**: The training and testing loss will be plotted to help you understand how the model is performing.
- **Accuracy Curve**: The training and testing accuracy will be plotted as well, giving you a clear indication of how well the model is learning.

### Note:
- The code will automatically use the results from the previous training process to generate these plots.
- The graphs will help you assess the model’s performance throughout the training process.

Simply run the cell, and you should see the visualizations appear!


In [ ]:
visualize_training(results)

## 3.4 Analyze the results

Now that you have trained your model, it’s time to analyze its performance. Based on the results, you may notice that the model is either overfitting or underfitting. In either case, there are specific actions you can take to improve the model.

### Does the model overfit or underfit?

- **Overfitting**: If your model's training accuracy is much higher than the test accuracy, this may indicate overfitting. In that case, you can try the following approaches:
    1. **Add Dropout Layers**: Insert `nn.Dropout` layers between the linear layers in your model to help prevent overfitting.
    2. **Regularization**: Apply L2 regularization to the optimizer by setting `weight_decay` to a non-zero value. This can help penalize large weights and reduce overfitting.
    3. **Decrease Learning Rate**: Try reducing the learning rate to make the model converge more smoothly and avoid overfitting.

- **Underfitting**: If both your training and testing accuracies are low and not improving, this may indicate underfitting. You can try the following:
    1. **Increase Model Complexity**: Add more layers to your model to make it more capable of learning complex patterns. Start by adding a linear layer that transforms the input into a 512-dimensional space, and slowly build up to more complex layers until reaching the output layer.
    2. **Increase Training Epochs**: You can try running for more epochs to give the model more time to learn.
    3. **Tweak the Architecture**: Experiment with different activation functions, such as ReLU, Leaky ReLU, or ELU, or add more hidden layers to capture more complex features in the data.

### Steps for Adjusting the Model:
1. **If you observe overfitting**, start by adding Dropout layers or apply regularization via the optimizer.
2. **If you observe underfitting**, try making the model deeper by adding more layers or adjusting the learning rate and training epochs.

### The goal:
Your goal here is to strike a balance: you want the model to learn well on the training data without overfitting to it, while also performing well on the test data.

Try these suggestions, and run the training again to see if the changes improve the performance.

Good luck!


Discuss the model's performance here:

# 4. Improving the Model

Now that you've analyzed the model's performance, you may need to modify the architecture to address overfitting or underfitting. In this step, you will:

1. **Create a new model with suggested adjustments**.

2. **Initialize the model** and place it on the correct device.

3. **Define the loss function** and **optimizer**.

4. **Train the model**.

5. **Visualize the results**.


# 5. Plot predictions

## 5.1 model_1 predictions

In [ ]:
import random

# Plot the first X test images, their predicted labels, and the true labels.
# Color correct predictions in blue and incorrect predictions in red.
num_rows = 5
num_cols = 3
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))

# Convert dataloader into a list of batches
all_batches = list(test_dataloader)

# Pick a random batch
X_batch, y_batch = random.choice(all_batches)
X_batch, y_batch = X_batch.to(device), y_batch.to(device)

# Get model predictions
model_1.eval()  # Ensure model is in eval mode
with torch.no_grad():
    y_pred = model_1(X_batch)  # Get raw logits
    y_pred_prob = torch.softmax(y_pred, dim=1)  # Convert logits to probabilities
    y_pred_labels = torch.argmax(y_pred_prob, dim=1)  # Get predicted labels

for i in range(num_images):
    plt.subplot(num_rows, 2*num_cols, 2*i+1)
    plot_image(i, y_pred_prob[i].cpu(), y_batch.cpu(), X_batch.squeeze().cpu())
    plt.subplot(num_rows, 2*num_cols, 2*i+2)
    plot_value_array(i, y_pred_prob[i].cpu(), y_batch.cpu())
plt.tight_layout()
plt.show()

## 5.2 the improved model predictions

In [ ]:
# >>> ENTER YOUR CODE HERE <<<

# 6. Save and load the best performing model

Let's finish this section off by saving and loading in our best performing model.</p>
<p>We can save and load a PyTorch model using a combination of:</p>
<ul>
<li><code>torch.save</code> - a function to save a whole PyTorch model or a model's <code>state_dict()</code>.</li>
<li><code>torch.load</code> - a function to load in a saved PyTorch object.</li>
<li><code>torch.nn.Module.load_state_dict()</code> - a function to load a saved <code>state_dict()</code> into an existing model instance.</li>
</ul>
<p>You can see more of these three in the <a href="https://pytorch.org/tutorials/beginner/saving_loading_models.html">PyTorch saving and loading models documentation</a>.</p>
<p>For now, let's save our <code>model_1</code>'s <code>state_dict()</code> then load it back in and evaluate it to make sure the save and load went correctly.</p>

In [ ]:
from pathlib import Path

# Create models directory (if it doesn't already exist), see: https://docs.python.org/3/library/pathlib.html#pathlib.Path.mkdir
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, # create parent directories if needed
                 exist_ok=True # if models directory already exists, don't error
)

# Create model save path
MODEL_NAME = "model.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

# Save the model state dict
print(f"Saving model to: {MODEL_SAVE_PATH}")
# >>> ENTER YOUR CODE HERE <<<
torch.save(, # only saving the state_dict() only saves the learned parameters
           f=MODEL_SAVE_PATH)

# 6. **Resources**

<li><a href="https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html">CIFAR10 Tutorial</a></li>
<li><a href="https://www.learnpytorch.io/">Learn PyTorch for Deep Learning: Zero to Mastery book</a></li>
<li><a href="https://lightning.ai/docs/torchmetrics/stable/classification/accuracy.html">Accuracy</a></li>
<li><a href="https://onnxruntime.ai/docs/tutorials/iot-edge/rasp-pi-cv.html">ONNX Runtime IoT Deployment on Raspberry Pi</a></li>
<li><a href="https://pytorch.org/vision/main/generated/torchvision.datasets.CIFAR10.html">CIFAR10</a></li>